In [2]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F


In [3]:
scholar_df = pd.DataFrame({
    "scholar_id": [1, 2, 3, 4],

    "research_interests": [
        "deep learning for flood prediction and hydrological modelling",
        "computer vision and medical image analysis",
        "structural health monitoring using machine learning",
        "remote sensing and climate change"
    ],

    "expertise": [
        "deep learning, hydrology, remote sensing",
        "computer vision, deep learning, medical imaging",
        "machine learning, structural engineering, sensors",
        "remote sensing, GIS, climate modelling"
    ],

    "department": [
        "Civil Engineering",
        "Computer Science",
        "Civil Engineering",
        "Civil Engineering"
    ],

    "university": [
        "IIT Kharagpur",
        "IIT Delhi",
        "IIT Bombay",
        "IIT Kharagpur"
    ],

    "country": [
        "India",
        "India",
        "India",
        "India"
    ],

    "publication_count": [
        12, 25, 18, 30
    ],

    "citation_count": [
        120, 450, 200, 600
    ],

    "years_experience": [
        2, 5, 3, 7
    ]
})

scholar_df

,scholar_id,research_interests,expertise,department,university,country,publication_count,citation_count,years_experience
0,1,deep learning for flood prediction and hydrolo...,"deep learning, hydrology, remote sensing",Civil Engineering,IIT Kharagpur,India,12,120,2
1,2,computer vision and medical image analysis,"computer vision, deep learning, medical imaging",Computer Science,IIT Delhi,India,25,450,5
2,3,structural health monitoring using machine lea...,"machine learning, structural engineering, sensors",Civil Engineering,IIT Bombay,India,18,200,3
3,4,remote sensing and climate change,"remote sensing, GIS, climate modelling",Civil Engineering,IIT Kharagpur,India,30,600,7


In [4]:
categorical_columns = ["department", "country", "university"]
categorical_columns

['department', 'country', 'university']

In [5]:
categorical_maps = {}

for col in categorical_columns:

    values = scholar_df[col].fillna("UNKNOWN").astype(str)

    unique_values = sorted(values.unique())

    # mapping  = {value: idx for idx, value in enumerate(unique_values)}

    mapping = {
        value: idx
        for idx, value in enumerate(unique_values)
    }

    # Reserve 0 for UNKNOWN
    mapping = {
        value: idx + 1
        for idx, value in enumerate(unique_values)
    }

    mapping["UNKNOWN"] = 0

    categorical_maps[col] = mapping

categorical_maps

{'department': {'Civil Engineering': 1, 'Computer Science': 2, 'UNKNOWN': 0},
 'country': {'India': 1, 'UNKNOWN': 0},
 'university': {'IIT Bombay': 1,
  'IIT Delhi': 2,
  'IIT Kharagpur': 3,
  'UNKNOWN': 0}}

In [ ]:
# Convert categories → IDs
categorical_ids = {}

for col in categorical_columns:

    mapping = categorical_maps[col]

    categorical_ids[col] = (
        scholar_df[col]
        .fillna("UNKNOWN")
        .astype(str)
        .map(lambda x: mapping.get(x, 0))
        .values
    )

In [7]:
categorical_ids

{'department': array([1, 2, 1, 1]),
 'country': array([1, 1, 1, 1]),
 'university': array([3, 2, 1, 3])}

In [8]:
device = torch.device(
    "cude" if torch.cuda.is_available() else "cpu"
)
device

device(type='cpu')

In [ ]:
# Convert categorical IDs to tensors
categorical_tensors = {}

for col in categorical_columns:
    mapping = categorical_ids[col]
    # print(mapping)
    categorical_tensors[col] = torch.tensor(
        categorical_ids[col],
        dtype=torch.long,
        device=device
    )
categorical_tensors

[1 2 1 1]
[1 1 1 1]
[3 2 1 3]


{'department': tensor([1, 2, 1, 1]),
 'country': tensor([1, 1, 1, 1]),
 'university': tensor([3, 2, 1, 3])}

In [ ]:
# Ids -> embedding (num_categories.X.32)
class CategoricalEncoder(nn.Module):
    def __init__(self, categorical_maps, embedding_dim=32):
        super().__init__()
        self.embeddings = nn.ModuleDict()

        for col, mapping in categorical_maps.items():
            num_categories = len(mapping)
            self.embeddings[col] = nn.Embedding(
                num_embeddings=num_categories,
                embedding_dim=embedding_dim,
                padding_idx=0
            ) 

    def forward(self, categorical_inputs):
        outputs = []

        for col, embedding in self.embeddings.items():
            # print(embedding)
            x = categorical_inputs[col]
            emb = embedding(x)
            outputs.append(emb)
        return torch.cat(outputs, dim=1)

In [22]:
categorical_encoder = CategoricalEncoder(categorical_maps, embedding_dim=32).to(device)
categorical_encoder

CategoricalEncoder(
  (embeddings): ModuleDict(
    (department): Embedding(3, 32, padding_idx=0)
    (country): Embedding(2, 32, padding_idx=0)
    (university): Embedding(4, 32, padding_idx=0)
  )
)

In [23]:
categorical_embedding = categorical_encoder(
    categorical_tensors
)
print(categorical_embedding.shape)

Embedding(3, 32, padding_idx=0)
Embedding(2, 32, padding_idx=0)
Embedding(4, 32, padding_idx=0)
torch.Size([4, 96])


In [ ]:
"""self.embeddings["department"] = nn.Embedding(
                        num_embeddings=3,
                        embedding_dim=32,
                        padding_idx=0
                    )

                                                32 values
                        ↓ ↓ ↓ ↓ ↓ ↓

            ID 0        [................]  UNKNOWN
            ID 1        [................]  Civil Engineering
            ID 2        [................]  Computer Science

                ID 0 → [32 numbers] UNKNOWN
                ID 1 → [32 numbers] Civil Engineering
                ID 2 → [32 numbers] Computer Science
            """
"""
self.embeddings = {

        "department":
            DepartmentEmbeddingLayer,

        "university":
            UniversityEmbeddingLayer,

        "country":
            CountryEmbeddingLayer
}


----->>>>>   
categorical_inputs:

{
    "department": tensor([1, 2, 1, 1]),
    "university": tensor([3, 2, 1, 3]),
    "country": tensor([1, 1, 1, 1])
}


--->>> 
x = [1, 2, 1, 1]
embedding(x)
Scholar 1 → row 1 → [b1 ... b32]
Scholar 2 → row 2 → [c1 ... c32]
Scholar 3 → row 1 → [b1 ... b32]
Scholar 4 → row 1 → [b1 ... b32]

emb.shape = [4, 32]
"""